# Dynamic windfield --- **peaks + animation frames in ONE run** (A100, Google Drive)

The single notebook to run for a design's dynamic-Powell data. `--full` marches each
storm **once** and writes BOTH the peak checkpoints (the static footprint the map draws)
AND all four products' animation frames --- because the peak is just the running max over
those very frames, there is no reason to march twice. This replaces the old
peaks-then-frames two-notebook workflow.

Everything is written to **Google Drive** (`/content/drive/MyDrive/mm_dynamic`), so a
disconnect never loses work: each finished batch writes its `.bin` frames and a peak
checkpoint, and a reconnect **resumes** (batches whose checkpoint exists are skipped).

**For a NEW design**, `colab_bundle.zip` needs only: `pipeline/` code, `outputs/web/grid
.json`, `outputs/web/roughness.json`, and the design's `inputs_<design>.json` --- NOT any
`powell_dyn_*` products (this run computes them).

Outputs on Drive: `powell_dyn_constrained*.json` (4 products) + `dyn_frames_constrained/`
(200 x 4 x 60 KB `.bin`) + `dyn_frames_constrained.json` manifest.

**Before running:** `Runtime -> Change runtime type -> A100 GPU`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU --- set Runtime to A100')

## 2. Mount Drive & set up / resume the working dir
First run: pick `colab_bundle.zip` when prompted. Reconnecting later: it detects the
existing Drive copy and **resumes** --- no re-upload.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
if os.path.exists(os.path.join(MM, 'pipeline', 'windfield_dynamic_batch.py')):
    print('Working dir already on Drive --- will RESUME:', MM)
    ck = os.path.join(MM, 'outputs/dynamic/precompute_constrained')
    nck = len([f for f in os.listdir(ck) if f.endswith('.json')]) if os.path.exists(ck) else 0
    fdir = os.path.join(MM, 'outputs/web/dyn_frames_constrained')
    nb = len([f for f in os.listdir(fdir) if f.endswith('.bin')]) if os.path.exists(fdir) else 0
    print(f'  {nck} batch checkpoint(s), {nb} frame .bin already on Drive '
          f'(targets: 8 checkpoints, 800 .bin).')
else:
    os.makedirs(MM, exist_ok=True)
    from google.colab import files
    up = files.upload()  # choose colab_bundle.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(MM)
    print('bundle extracted to Drive:', MM)

## 3. Choose what to run
`--full` runs the marine pass + the B/C/D marches for every selected storm and keeps both
peaks and frames --- so cost is the full dynamic run (~2 h for all 200 on the A100). Fully
resumable: batches whose checkpoint exists are skipped.

- `'all'` --- all 200 storms (cost-sorted, cheap first)
- `'bulk'` --- the cheap 155 · `'tail'` --- the expensive 45 · `'vec:75,90'` --- specific storms

`BATCH_SIZE`: storms marched together. 25 is safe; on the A100's 40 GB, larger batches with
`'bulk'`/`'tail'` amortize per-step overhead (keep batches cost-homogeneous). NOTE frames
add GPU memory (4 products x batch x 840 x 73 floats), so keep `--full` batches modest
(25--50) unless you have headroom.

In [ ]:
SELECT = 'all'      # '' | 'all' | 'bulk' | 'tail' | 'vec:75,90'
BATCH_SIZE = 25     # frames use extra GPU mem under --full; 25 is safe on the A100

## 4. Run (writes peaks + frames to Drive as it goes)
Streams progress. **If it disconnects:** reconnect, re-run cells 1--2, then this cell --- it
skips finished batches and resumes.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import subprocess, sys
cmd = [sys.executable, '-u', 'pipeline/windfield_dynamic_batch.py',
       '--constrained', '--full', '--select', SELECT, '--batch-size', str(BATCH_SIZE)]
print('running:', ' '.join(cmd), '\n')
p = subprocess.Popen(cmd, cwd=MM, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if not any(w in line for w in ('Warning','warnings.warn','ConvergenceWarning')):
        print(line, end='')
p.wait()
print('\nexit code:', p.returncode)

## 5. Assemble products + manifest, package to Drive & download
Safe to run anytime (partial runs OK). Rebuilds the 4 products from the checkpoints, rewrites
the frame manifest, and zips **products + frames + manifest** to Drive, then downloads it.
Unzip into the repo's `outputs/web/` locally.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import subprocess, sys, os, glob, zipfile
# 1) build the 4 product JSONs from the checkpoints (applies C<=A, D<=B)
subprocess.run([sys.executable, 'pipeline/windfield_dynamic_batch.py',
                '--constrained', '--assemble-only'], cwd=MM)
# 2) rewrite the frame manifest from the existing .bin (no re-compute, no clobber)
sys.path.insert(0, os.path.join(MM, 'pipeline'))
import windfield_dynamic_batch as B
B.FRAMES_DIR = os.path.join(B.WEB, 'dyn_frames_constrained')
B.MANIFEST_JSON = os.path.join(B.WEB, 'dyn_frames_constrained.json')
B.PRODUCTS = {k: (fn.replace('.json', '_constrained.json'), note)
             for k, (fn, note) in B.PRODUCTS.items()}
B.write_frame_manifest()
# 3) zip products + manifest + frames
prod = glob.glob(f'{MM}/outputs/web/powell_dyn_constrained*.json') + \
       glob.glob(f'{MM}/outputs/web/powell_dyn_*_constrained.json')
prod = sorted(set(prod))
man  = os.path.join(MM, 'outputs/web/dyn_frames_constrained.json')
bins = sorted(glob.glob(os.path.join(MM, 'outputs/web/dyn_frames_constrained', '*.bin')))
out  = '/content/drive/MyDrive/dynamic_full_constrained.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in prod + [man] + bins:
        z.write(f, os.path.relpath(f, MM))
print(f'{len(prod)} products + manifest + {len(bins)} frames -> {out} (also on Drive)')
from google.colab import files
files.download(out)

### If the runtime disconnects while you're away
Nothing is lost --- finished batches' checkpoints and `.bin` are on Drive. On return: re-run
**Cell 1**, **Cell 2** (resumes, no upload), then **Cell 4** to finish, or **Cell 5** to grab
what's done. `dynamic_full_constrained.zip` also stays on Drive.